<center><font size="6" color="#2D6A4F"><b>TransUNet Training for Speckle Pattern Analysis</b></font></center>


**Environment Setup (Libraries & Installation)**

In [ ]:
!git clone https://github.com/Beckschen/TransUNet.git

In [ ]:
!pip install -U timm
!pip install -U albumentations[imgaug,extra,pytorch]
!pip install -U transformers
!pip install -U pytorch-lightning
!pip install -U torchmetrics
!pip install -U optuna
!pip install ml-collections

In [ ]:
import os
import sys
import ast
import time
import math
import json
import shutil
import random
import logging
from datetime import datetime
from pathlib import Path
from typing import Tuple, Optional

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as transforms

import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule
from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    LearningRateMonitor,
)
from pytorch_lightning.loggers import CSVLogger

from torchmetrics import JaccardIndex
from torchmetrics.functional import (
    jaccard_index,
    precision_recall_curve,
    pearson_corrcoef,
)

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from timm.models.layers import trunc_normal_

from transformers import ViTModel, ViTConfig
from einops import rearrange

import ml_collections
import optuna

from networks.vit_seg_modeling import VisionTransformer, CONFIGS


seed_value = 11
os.environ["PYTHONHASHSEED"] = str(seed_value)

random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

pl.seed_everything(seed_value)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
sys.path.append("/content/TransUNet")


model_name = "TransUNet_B32"
base_log_dir = f"/content/drive/MyDrive/TranSpeckle/optuna_logs/{model_name}"
Path(base_log_dir).mkdir(parents=True, exist_ok=True)

**Loading and Preprocessing Function for Training and Validation Sets**

In [ ]:
def load_processed_train_valid_vit(dataset_name: str) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    base_path = Path('/content/drive/MyDrive/TranSpeckle/data_sets') / dataset_name

    def binning_2x2(image: np.ndarray) -> np.ndarray:
        return image.reshape((image.shape[0] // 2, 2, image.shape[1] // 2, 2)).mean(axis=(1, 3))

    def process_pair(x_dir: Path, y_dir: Path, label: str) -> Tuple[torch.Tensor, torch.Tensor]:
        x_list, y_list = [], []
        x_files = sorted(x_dir.glob('*.tif'))
        print(f"Found {len(x_files)} files in '{label}'")

        t0 = time.time()
        for i, x_file in enumerate(x_files):
            fname = x_file.name
            y_file = y_dir / fname
            if not y_file.exists():
                print(f"Missing GT for: {fname}")
                continue

            x = cv2.imread(str(x_file), cv2.IMREAD_UNCHANGED)
            if len(x.shape) == 3:
                x = cv2.cvtColor(x, cv2.COLOR_BGR2GRAY)

            x = binning_2x2(x)
            x = (x / np.max(x)).astype(np.float32) if np.max(x) > 0 else x.astype(np.float32)
            x = np.expand_dims(x, axis=0)

            y = cv2.imread(str(y_file), cv2.IMREAD_UNCHANGED)
            if len(y.shape) == 3:
                y = cv2.cvtColor(y, cv2.COLOR_BGR2GRAY)

            y = binning_2x2(y)
            y_binary = (y > 0).astype(np.uint8)
            y_tensor = torch.from_numpy(y_binary).long()
            y_onehot = F.one_hot(y_tensor, num_classes=2).permute(2, 0, 1).float()

            x_list.append(torch.from_numpy(x))
            y_list.append(y_onehot)

            if i % 100 == 0 and i > 0:
                print(f"Processed {i}/{len(x_files)} files in '{label}'")

        print(f"Done loading '{label}' in {time.time() - t0:.1f} seconds")
        return torch.stack(x_list), torch.stack(y_list)

    print("Starting to load and preprocess training and validation sets...")
    x_train, y_train = process_pair(base_path / 'x_train', base_path / 'y_train', 'train')
    x_valid, y_valid = process_pair(base_path / 'x_valid', base_path / 'y_valid', 'valid')

    print(f"Loaded all data from '{dataset_name}' — train: {x_train.shape[0]}, valid: {x_valid.shape[0]}")
    return x_train, y_train, x_valid, y_valid

In [ ]:
def load_processed_test_vit(dataset_name: str) -> Tuple[torch.Tensor, torch.Tensor]:
    base_path = Path('/content/drive/MyDrive/TranSpeckle/data_sets') / dataset_name
    x_path = base_path / 'x_test'
    y_path = base_path / 'y_test'

    x_list, y_list = [], []
    x_files = sorted(x_path.glob('*.tif'))
    print(f"\nStarting to load and preprocess test set from '{dataset_name}'")
    print(f"Found {len(x_files)} test files")

    t0 = time.time()
    for i, x_file in enumerate(x_files):
        fname = x_file.name
        y_file = y_path / fname

        if not y_file.exists():
            print(f"Missing GT for: {fname}")
            continue

        x = cv2.imread(str(x_file), cv2.IMREAD_UNCHANGED)
        if len(x.shape) == 3:
          x = cv2.cvtColor(x, cv2.COLOR_BGR2GRAY)

        x = binning_2x2(x)
        x = (x / np.max(x)).astype(np.float32) if np.max(x) > 0 else x.astype(np.float32)
        x = np.expand_dims(x, axis=0)

        y = cv2.imread(str(y_file), cv2.IMREAD_UNCHANGED)
        if len(y.shape) == 3:
                y = cv2.cvtColor(y, cv2.COLOR_BGR2GRAY)

        y = binning_2x2(y)
        y_binary = (y > 0).astype(np.uint8)
        y_tensor = torch.from_numpy(y_binary).long()
        y_onehot = F.one_hot(y_tensor, num_classes=2).permute(2, 0, 1).float()

        x_list.append(torch.from_numpy(x))
        y_list.append(y_onehot)

        if i % 50 == 0 and i > 0:
            print(f"Processed {i}/{len(x_files)} files")

    print(f"Done loading test set in {time.time() - t0:.1f} seconds")
    print(f"Total loaded: {len(x_list)} examples")
    return torch.stack(x_list), torch.stack(y_list)


**Data Augmentation**

In [ ]:
class AugmentedSpeckleDataset(Dataset):
    def __init__(self, x_tensor, y_tensor, transform):
        self.x_tensor = x_tensor
        self.y_tensor = y_tensor
        self.transform = transform

    def __len__(self):
        return self.x_tensor.size(0)

    def __getitem__(self, idx):
        x = self.x_tensor[idx]
        y = self.y_tensor[idx]

        x = x.permute(1, 2, 0).numpy()
        y = y.permute(1, 2, 0).numpy()

        augmented = self.transform(image=x, mask=y)
        x = augmented['image']
        y = augmented['mask']

        y = y.permute(2, 0, 1)

        return x.float(), y.float()

In [ ]:
class SpeckleDataModule(LightningDataModule):
    def __init__(
        self,
        train_data=None,
        val_data=None,
        test_data=None,
        batch_size: int = 16,
        num_workers: int = 4,
    ):
        super().__init__()
        self.train_data = train_data
        self.val_data = val_data
        self.test_data = test_data
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.train_transform = A.Compose([
            A.CropAndPad(percent=(-0.03, 0.0), border_mode=cv2.BORDER_CONSTANT, fill=0, keep_size=True, p=0.08),
            A.RandomBrightnessContrast(brightness_limit=0.08, p=0.05),
            A.Rotate(limit=10, p=0.07, border_mode=0,),
            A.ToTensorV2()
        ])

    def setup(self, stage: Optional[str] = None):
        if stage == 'fit' or stage is None:
            x_train, y_train = self.train_data
            x_val, y_val = self.val_data

            self.train_dataset = AugmentedSpeckleDataset(x_train, y_train, transform=self.train_transform)
            self.val_dataset = TensorDataset(x_val, y_val)

        if stage == 'test' or stage is None:
            x_test, y_test = self.test_data
            self.test_dataset = TensorDataset(x_test, y_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

**Orientation-Aware Positional Embeddings**

In [ ]:
class OrientationPositionalEmbedding(nn.Module):
    def __init__(self, base_embedding: nn.Parameter, emb_dim, grid_size=16, add_theta=True):
        super().__init__()
        self.emb_dim = emb_dim
        self.add_theta = add_theta
        self.base_embedding = base_embedding

        if add_theta:
            self.theta_proj = nn.Linear(2, emb_dim)
            self.register_buffer("theta_encoding", self._create_theta_encoding(grid_size))

    def _create_theta_encoding(self, grid_size):
        encoding = torch.zeros(grid_size * grid_size, 2)
        center = grid_size / 2 - 0.5
        for i in range(grid_size):
            for j in range(grid_size):
                dx = j - center
                dy = i - center
                theta = math.atan2(dy, dx)
                idx = i * grid_size + j
                encoding[idx, 0] = math.sin(theta)
                encoding[idx, 1] = math.cos(theta)
        return encoding

    def forward(self, x):
        B, N, D = x.shape
        pos = self.base_embedding.repeat(B, 1, 1)

        if self.add_theta:
            theta_feats = self.theta_proj(self.theta_encoding)
            theta_feats = theta_feats.unsqueeze(0).repeat(B, 1, 1)
            pos = pos + theta_feats

        return x + pos

In [ ]:
class TransUNetWithOrientationPE(VisionTransformer):
    def __init__(self, config, img_size=256, num_classes=2,
                 zero_head=True, vis=False,
                 add_theta=True,
                 use_pretrained=True,
                 pretrained_ckpt=None):
        super().__init__(config, img_size=img_size, num_classes=num_classes,
                         zero_head=zero_head, vis=vis)


        if pretrained_ckpt and os.path.isfile(pretrained_ckpt):
            try:
                state_dict = torch.load(pretrained_ckpt, map_location="cpu")

                if "model" in state_dict:
                    state_dict = state_dict["model"]

                loaded_keys = state_dict.keys()

                model_keys = dict(self.named_parameters()).keys()

                matched_keys = list(set(loaded_keys).intersection(model_keys))


                match_ratio = len(matched_keys) / len(model_keys) * 100

                self.load_state_dict(state_dict, strict=False)

                print(f"[INFO] Loaded pretrained weights from: {pretrained_ckpt}")
                print(f"[INFO] Matched {len(matched_keys)} keys — Match ratio: {match_ratio:.2f}%")

            except Exception as e:
                print(f"[WARNING] Failed to load pretrained weights from {pretrained_ckpt}")
                print(f"[ERROR] {e}")
        else:
            print(f"[INFO] No pretrained weights loaded. Training from scratch.")

        self.img_size = img_size
        self.add_theta = add_theta

        patch_size = config.patches.size[0]
        num_patches = (img_size // patch_size) ** 2
        embed_dim = config.hidden_size

        base_embedding = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        trunc_normal_(base_embedding, std=0.02)

        self.orientation_pos_embed = OrientationPositionalEmbedding(
            base_embedding=base_embedding,
            emb_dim=embed_dim,
            grid_size=int(num_patches ** 0.5),
            add_theta=add_theta
        )

    def forward(self, x):
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)

        x, features = self.transformer.embeddings(x)
        x = self.orientation_pos_embed(x)

        x, attn_weights = self.transformer.encoder(x)

        x = self.decoder(x, features)

        logits = self.segmentation_head(x)
        return logits

**TransUNet Lightning Module**

In [ ]:
class LitTransUNet(pl.LightningModule):
    def __init__(self, lr=1e-4, use_orientation_pos_emb=False,
                 use_pretrained=True, pretrained_ckpt=None):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr

        config = CONFIGS["ViT-B_32"]
        config.n_classes = 2
        config.n_skip = 0


        self.model = TransUNetWithOrientationPE(
        config=config,
        img_size=256,
        num_classes=2,
        pretrained_ckpt=pretrained_ckpt if use_pretrained else None,
        zero_head=True,
        vis=False)

        self.criterion = self.balanced_cross_entropy
        self.loss_fn = nn.CrossEntropyLoss()
        self.iou = JaccardIndex(task="binary", num_classes=2)

    def forward(self, x):
        return self.model(x)

    def balanced_cross_entropy(self, pred, target, eps=1e-6):
        pred_softmax = torch.softmax(pred, dim=1).clamp(min=eps, max=1 - eps)
        loss = - (target * torch.log(pred_softmax) + (1 - target) * torch.log(1 - pred_softmax))
        return loss.mean()

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        if logits.shape[2:] != y.shape[2:]:
          logits = F.interpolate(
              logits,
              size=y.shape[2:],
              mode="bilinear",
              align_corners=False
          )
        loss = self.criterion(logits, y)
        preds = (logits[:,1]>0.5).long()
        targets = y[:,1].long()
        iou = self.iou(preds, targets)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_iou", iou, prog_bar=True, on_step=False, on_epoch=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        if logits.shape[2:] != y.shape[2:]:
          logits = F.interpolate(
              logits,
              size=y.shape[2:],
              mode="bilinear",
              align_corners=False
          )
        loss = self.criterion(logits, y)
        preds = (logits[:,1]>0.5).long()
        targets = y[:,1].long()
        iou = self.iou(preds, targets)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_iou", iou, prog_bar=True, on_step=False, on_epoch=True, logger=True)

    def configure_optimizers(self):
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)

            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="max",
                factor=0.5,
                patience=8,
                threshold=1e-4,
                cooldown=2,
                min_lr=1e-6,
                verbose=True
            )

            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "monitor": "val_iou",
                    "interval": "epoch",
                    "frequency": 1,
                    "strict": True
                }
            }

**Optuna Objective: Hyperparameter Search**

In [ ]:
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.WARNING)

def objective(trial, dm, use_pretrained=True):
    lr = trial.suggest_float("lr", 2e-4, 4e-4, log=True)

    model = LitTransUNet(
        lr=lr,
        use_pretrained=None,
        pretrained_ckpt="/content/drive/MyDrive/pretrained/vit_b32.pth"
    )

    version_dir = f"trial_{trial.number}_{datetime.now().strftime('%d%m%Y')}"
    trial_log_dir = os.path.join(base_log_dir, "transunet_tuning", version_dir)
    os.makedirs(trial_log_dir, exist_ok=True)

    logger = CSVLogger(
        save_dir=trial_log_dir,
        name="",
        version="."
    )

    ckpt_temp_dir = os.path.join("/content/temp_checkpoints", version_dir)
    os.makedirs(ckpt_temp_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=ckpt_temp_dir,
        filename="best",
        monitor="val_iou",
        mode="max",
        save_top_k=1,
        save_on_train_epoch_end=True
    )

    early_stop = EarlyStopping(monitor="val_iou", patience=30, mode="max")

    trainer = pl.Trainer(
        max_epochs=350,
        callbacks=[checkpoint_callback, early_stop],
        logger=logger,
        enable_progress_bar=True,
        accelerator="auto"
    )

    trainer.fit(model, dm)

    best_ckpt_src = checkpoint_callback.best_model_path
    best_ckpt_dst = os.path.join(trial_log_dir, "checkpoints", "best.ckpt")
    os.makedirs(os.path.dirname(best_ckpt_dst), exist_ok=True)
    shutil.copy2(best_ckpt_src, best_ckpt_dst)

    trial.set_user_attr("best_model_path", best_ckpt_dst)

    print("-" * 60)
    print(f"\n Trial {trial.number} finished")
    print(f"   val_iou: {trainer.callback_metrics['val_iou'].item():.4f}")
    print(f"   Learning Rate: {lr}")
    print("-" * 60)

    return trainer.callback_metrics["val_iou"].item()

**Evaluation**

In [ ]:
@torch.no_grad()
def predict_testset_torch_cls(
    ckpt_path: str,
    model_class,
    x_test,
    batch_size: int = 16,
    out_size: tuple[int,int] = None,
    device: str | None = None,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    x = torch.as_tensor(x_test).float()
    if x.ndim == 3:
        x = x.unsqueeze(1)

    dl = DataLoader(TensorDataset(x), batch_size=batch_size, shuffle=False)
    model = model_class.load_from_checkpoint(ckpt_path).to(device).eval()

    preds = []
    for (xb,) in dl:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb)
        if out_size is not None and logits.shape[2:] != out_size:
            logits = F.interpolate(logits, size=out_size, mode="bilinear", align_corners=False)
        probs = F.softmax(logits, dim=1).permute(0, 2, 3, 1).cpu()  # [B,H,W,2]
        preds.append(probs)
    return torch.cat(preds, dim=0).numpy()

In [ ]:
def evaluate_predictions_torchmetrics(y_pred: np.ndarray, y_true: np.ndarray, threshold: float = 0.5):
    if y_true.ndim == 4 and y_true.shape[-1] == 2:
        y_true_bin = np.argmax(y_true, axis=-1).astype(np.int32)
    else:
        y_true_bin = y_true.astype(np.int32)

    p1 = y_pred[..., 1]
    y_pred_bin = (p1 >= threshold).astype(np.int32)

    preds_t = torch.from_numpy(y_pred_bin)
    target_t = torch.from_numpy(y_true_bin)
    iou = jaccard_index(preds_t, target_t, task="binary").item()

    p1_t = torch.from_numpy(p1).float()
    gt_t = torch.from_numpy(y_true_bin).float()
    per_img_pcc = []
    for i in range(p1_t.shape[0]):
        per_img_pcc.append(pearson_corrcoef(p1_t[i].flatten(), gt_t[i].flatten()).item())
    pcc_mean = float(np.mean(per_img_pcc))

    print(f"IoU (mean over set): {iou:.3f}")
    print(f"PCC (Pearson, per-image mean): {pcc_mean:.3f}")

    return {
        "iou_mean": iou,
        "pcc_mean": pcc_mean,
        "pcc_per_image": np.array(per_img_pcc, dtype=np.float32),
    }

In [ ]:
def get_best_ckpt_path(base_log_dir: str, model_name: str):
    db_file_path = Path(base_log_dir) / "study.db"
    storage_path = f"sqlite:////{db_file_path}"

    try:
        study = optuna.load_study(study_name=f"{model_name}_study", storage=storage_path)
        best = study.best_trial
        cand = best.user_attrs.get("best_model_path") or best.user_attrs.get("last_model_path")
        if cand and os.path.exists(cand):
            return cand, best.number, best.value
    except Exception as e:
        print(f"[WARN] Could not load study or attrs: {e}")

    base = Path(base_log_dir)
    candidates = list(base.rglob("continued_checkpoints/best.ckpt")) \
              or list(base.rglob("checkpoints/best.ckpt"))
    if candidates:
        best_file = max(candidates, key=lambda p: p.stat().st_mtime)
        return str(best_file), None, None

    raise FileNotFoundError("No checkpoint found under base_log_dir")

**Visualize model prediction vs. ground truth**


In [ ]:
def show_prediction_example_pt(x_test, y_true, y_pred, index=0, threshold: float = 0.5):

    def to_np(a):
        return a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a

    x = to_np(x_test)
    yt = to_np(y_true)
    yp = to_np(y_pred)

    xi, yi, pi = x[index], yt[index], yp[index]

    speckle = xi.squeeze()
    if speckle.ndim == 3:
        speckle = speckle[0] if speckle.shape[0] <= 3 else speckle[..., 0]

    if yi.ndim == 3 and yi.shape[0] == 2:
        gt_mask = np.argmax(yi, axis=0)
    elif yi.ndim == 3 and yi.shape[-1] == 2:
        gt_mask = np.argmax(yi, axis=-1)
    else:
        gt_mask = yi.squeeze().astype(np.int64)

    if pi.ndim == 3 and pi.shape[0] == 2:
        pred_mask = np.argmax(pi, axis=0)
    elif pi.ndim == 3 and pi.shape[-1] == 2:
        pred_mask = np.argmax(pi, axis=-1)
    else:
        pred_mask = (pi.squeeze() >= threshold).astype(np.int64)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(speckle, cmap='gray')
    plt.title("Speckle Input")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Prediction")
    plt.axis('off')

    plt.suptitle(f"Example {index}", fontsize=14)
    plt.tight_layout()
    plt.show()

<center><font size="5" color="#2D6A4F"><b>Model2400</b></font></center>

**Save/Load Processed Data + Lightning DataModule**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
x_train, y_train, x_val, y_val = load_processed_train_valid_vit("2400")

save_path = "/content/drive/MyDrive/TranSpeckle/cache/2400/processed.pt"

import os
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    'x_train': x_train,
    'y_train': y_train,
    'x_valid': x_val,
    'y_valid': y_val
}, save_path)

print(f"Saved to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/2400/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Automated Training & Hyperparameter Search (Optuna Study)**

In [ ]:
### Optional: Retrain or continue hyperparameter tuning

# By default, this notebook uses the provided best pre-trained weights.

# If you wish to retrain the model or continue hyperparameter optimization,
# Optuna will resume from the existing study database if available.
# Otherwise, skip this step and load the best pre-trained weights for evaluation and prediction.


'''
db_file_path = Path(base_log_dir) / "study.db"
storage_path = f"sqlite:////{db_file_path}"

load_existing = db_file_path.exists()

study = optuna.create_study(
    study_name=f"{model_name}_study",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=seed_value),
    storage=storage_path,
    load_if_exists=load_existing
)

start_time = time.time()

study.optimize(lambda trial: objective(trial, dm, use_pretrained=True), n_trials=10)

end_time = time.time()
total_minutes = (end_time - start_time) / 60
print(f"\nTotal optimization time: {int(total_minutes // 60)}h {int((total_minutes % 60))}m {int((total_minutes * 60) % 60)}s")

best = study.best_trial
print("\nBest_trial")
print(f"Trial {best.number} — val_iou: {best.value:.4f} — params: {best.params}")
'''

**Resume Training from Latest Checkpoint & Update Optuna Study**

In [ ]:
### Optional: Continue training the best Optuna trial

# This cell resumes training for the best trial from an existing Optuna study to allow additional epochs,
# since the initial tuning run used a limited epoch budget and may not have reached early stopping.
# Skip this cell to use the provided pre-trained weights for evaluation/inference.

'''
def pick_resume_ckpt(trial_dir: str, user_last: str | None, user_best: str | None):
    cc_last = os.path.join(trial_dir, "continued_checkpoints", "last.ckpt")
    for p in (cc_last, user_last, user_best):
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError("No resume checkpoint found (cc_last/user_last/user_best).")

db_file_path = Path(base_log_dir) / "study.db"
storage_path = f"sqlite:////{db_file_path}"
study = optuna.load_study(study_name=f"{model_name}_study", storage=storage_path)
print(f"[INFO] Loaded study with {len(study.trials)} trials.")

best = study.best_trial
best_path = best.user_attrs.get("best_model_path")
last_path = best.user_attrs.get("last_model_path")
assert best_path and os.path.exists(best_path), "best.ckpt was not found."

trial_dir = os.path.dirname(os.path.dirname(best_path))
resume_path = pick_resume_ckpt(trial_dir, last_path, best_path)

old_best_value = best.user_attrs.get("best_value", best.value)
old_best_ckpt = best.user_attrs.get("best_model_path", best_path)
try:
    old_ckpt_info = torch.load(old_best_ckpt, map_location="cpu")
    old_best_epoch = old_ckpt_info.get("epoch", "N/A")
except Exception:
    old_best_epoch = "N/A"

try:
    resume_info = torch.load(resume_path, map_location="cpu")
    resume_epoch = resume_info.get("epoch", "N/A")
except Exception:
    resume_epoch = "N/A"

print(f"[INFO] Previous best epoch: {old_best_epoch}")
print(f"[INFO] Previous best val_iou: {old_best_value:.6f}")
print(f"[INFO] Resuming from: {resume_path} | ckpt epoch: {resume_epoch}")


model = LitTransUNet(
        use_pretrained=None,
        lr=0.00020177966850933734
    )


version_dir = os.path.basename(trial_dir)
base_trial_root = os.path.dirname(trial_dir)
logger = CSVLogger(save_dir=base_trial_root, name=version_dir, version=".")

ckpt_temp_dir = os.path.join("/content/temp_checkpoints", version_dir)
os.makedirs(ckpt_temp_dir, exist_ok=True)

checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_temp_dir,
    filename="best",
    monitor="val_iou",
    mode="max",
    save_top_k=1,
    save_last=True,
    save_on_train_epoch_end=True
)

early_stop = EarlyStopping(monitor="val_iou", patience=24, mode="max", min_delta=1e-4)

trainer = pl.Trainer(
    max_epochs=800,
    callbacks=[checkpoint_callback, early_stop],
    logger=logger,
    enable_progress_bar=True,
    accelerator='auto',
    check_val_every_n_epoch=1,
    num_sanity_val_steps=0
)

print("[INFO] Continuing training...")
trainer.fit(model, dm, ckpt_path=resume_path)

continued_dir = os.path.join(trial_dir, "continued_checkpoints")
os.makedirs(continued_dir, exist_ok=True)

def _copy(src, dst):
    if src and os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f"[INFO] Copied: {src} -> {dst}")
        return dst
    print(f"[WARN] Not found, skip copy: {src}")
    return None

new_best_src = checkpoint_callback.best_model_path
new_best_dst = _copy(new_best_src, os.path.join(continued_dir, "best.ckpt"))

new_last_src = os.path.join(ckpt_temp_dir, "last.ckpt")
new_last_dst = _copy(new_last_src, os.path.join(continued_dir, "last.ckpt"))

if new_best_dst:
    best.set_user_attr("best_model_path", new_best_dst)
if new_last_dst:
    best.set_user_attr("last_model_path", new_last_dst)
new_best_score = None
if checkpoint_callback.best_model_score is not None:
    try:
        new_best_score = float(checkpoint_callback.best_model_score.cpu().item())
        best.set_user_attr("best_value", new_best_score)
    except Exception:
        pass

new_final_iou = float(trainer.callback_metrics['val_iou'].item()) if 'val_iou' in trainer.callback_metrics else None
print(f"[INFO] Continued run final val_iou: {new_final_iou:.6f}" if new_final_iou is not None else "[INFO] No final val_iou in metrics.")

print(f"[INFO] New best score: {checkpoint_callback.best_model_score}")
if (old_best_value is not None) and (checkpoint_callback.best_model_score is not None):
    delta = float(checkpoint_callback.best_model_score) - float(old_best_value)
    print(f"[INFO] IoU change: {old_best_value:.6f} -> {float(checkpoint_callback.best_model_score):.6f} (Δ {delta:+.6f})")

print(f"[INFO] best_model_path -> {best.user_attrs.get('best_model_path')}")
print(f"[INFO] last_model_path -> {best.user_attrs.get('last_model_path')}")
print(f"[INFO] best_value      -> {best.user_attrs.get('best_value')}")
'''

**group1**

**Load and create predictions**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
x_test, y_test = load_processed_test_vit("group1")

save_path = "/content/drive/MyDrive/TranSpeckle/cache/group1/processed.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    "x_test1": x_test,
    "y_test1": y_test
}, save_path)

print(f"Saved test set to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/group1/processed.pt")
x_test1, y_test1 = data['x_test1'], data['y_test1']

**Evaluation**

In [ ]:
ckpt_path, trial_no, best_val = get_best_ckpt_path(base_log_dir, model_name)
print(f"[INFO] Using ckpt: {ckpt_path}")
if trial_no is not None:
    print(f"[INFO] From trial #{trial_no} | best val_iou={best_val:.4f}")

y_pred = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test1, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test1)

results = evaluate_predictions_torchmetrics(y_pred=y_pred, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])

In [ ]:
# Display a reconstruction example (index 0–1999)

show_prediction_example_pt(x_test1, y_test1, y_pred, index=0)

**group2**

**Load and create predictions**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.
'''
x_test, y_test = load_processed_test_vit("group2")


save_path = "/content/drive/MyDrive/TranSpeckle/cache/group2/processed.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    "x_test2": x_test,
    "y_test2": y_test
}, save_path)

print(f"Saved test set to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/group2/processed.pt")
x_test2, y_test2 = data['x_test2'], data['y_test2']

**Evaluation**

In [ ]:
ckpt_path, trial_no, best_val = get_best_ckpt_path(base_log_dir, model_name)
print(f"[INFO] Using ckpt: {ckpt_path}")
if trial_no is not None:
    print(f"[INFO] From trial #{trial_no} | best val_iou={best_val:.4f}")

y_pred2 = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test2, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test2)

results = evaluate_predictions_torchmetrics(y_pred=y_pred2, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])

In [ ]:
# Display a reconstruction example (index 0–199)

show_prediction_example_pt(x_test2, y_test2, y_pred2, index=0)

**group3**

**Load and create predictions**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
x_test, y_test = load_processed_test_vit("group3")

save_path = "/content/drive/MyDrive/TranSpeckle/cache/group3/processed.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    "x_test3": x_test,
    "y_test3": y_test
}, save_path)

print(f"saved test set to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/group3/processed.pt")
x_test3, y_test3 = data['x_test3'], data['y_test3']

**Evaluation**

In [ ]:
ckpt_path, trial_no, best_val = get_best_ckpt_path(base_log_dir, model_name)
print(f"[INFO] Using ckpt: {ckpt_path}")
if trial_no is not None:
    print(f"[INFO] From trial #{trial_no} | best val_iou={best_val:.4f}")

y_pred3 = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test3, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test3)

results = evaluate_predictions_torchmetrics(y_pred=y_pred3, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])

In [ ]:
# Display a reconstruction example (index 0–799)

show_prediction_example_pt(x_test3, y_test3, y_pred3, index=0)

**Training Runner with DataModule & Checkpoint Resume**

In [ ]:
#Trains fixed runs using the best hyperparameters selected in the initial experiment,
#applied consistently across datasets of different sizes.

def train_fixed_run_dm(
    *,
    model_name: str,
    dataset_tag: str,
    model_class,
    hparams: dict,
    dm,
    base_log_dir: str = "/content/drive/MyDrive/TranSpeckle/optuna_logs",
    max_epochs: int = 350,
    patience: int = 35,
    resume: str = "fresh",
    init_ckpt_path: str | None = None
):
    import os, json, shutil, time, torch
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
    from pytorch_lightning.loggers import CSVLogger

    base_log_dir = os.path.join(base_log_dir, model_name)
    run_name = f"{model_name}_{dataset_tag}"
    run_dir  = os.path.join(base_log_dir, "fixed_runs", run_name)
    os.makedirs(run_dir, exist_ok=True)

    export_ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(export_ckpt_dir, exist_ok=True)

    tmp_ckpt_dir = os.path.join("/content/temp_checkpoints", run_name)
    os.makedirs(tmp_ckpt_dir, exist_ok=True)

    logger = CSVLogger(save_dir=run_dir, name="", version=".")
    lrmon  = LearningRateMonitor(logging_interval="epoch")

    ckpt_cb = ModelCheckpoint(
        dirpath=tmp_ckpt_dir,
        monitor="val_iou",
        mode="max",
        save_top_k=1,
        save_last=True,
        save_on_train_epoch_end=False
    )

    early = EarlyStopping(monitor="val_iou", mode="max", patience=patience, min_delta=1e-4)

    model = model_class(**hparams)

    resume_ckpt = None
    last_existing = os.path.join(export_ckpt_dir, f"last_{model_name}_{dataset_tag}.ckpt")

    if resume == "last" and os.path.exists(last_existing):
        resume_ckpt = last_existing
    elif resume == "from_path":
        assert init_ckpt_path and os.path.exists(init_ckpt_path), "init_ckpt_path not found."
        resume_ckpt = init_ckpt_path
    elif resume == "fresh":
        resume_ckpt = None

    if resume_ckpt:
        try:
            info = torch.load(resume_ckpt, map_location="cpu")
            print(f"[INFO] Resuming from: {resume_ckpt} | epoch in ckpt: {info.get('epoch','N/A')}")
        except Exception as e:
            print(f"[WARN] Could not read epoch from {resume_ckpt}: {e}")

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        callbacks=[ckpt_cb, early, lrmon],
        logger=logger,
        accelerator="auto",
        enable_progress_bar=True,
        check_val_every_n_epoch=1,
        num_sanity_val_steps=0
    )

    print(f"[INFO] Run: {run_name} | Strategy: {resume}")
    t0 = time.time()
    trainer.fit(model, dm, ckpt_path=resume_ckpt)
    t1 = time.time()
    print(f"[INFO] Training time: {(t1 - t0)/60:.1f} min")

    def _copy(src, dst):
        if src and os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"[INFO] Saved: {dst}")
            return dst
        print(f"[WARN] Missing: {src}")
        return None

    last_out = _copy(os.path.join(tmp_ckpt_dir, "last.ckpt"),
                     os.path.join(export_ckpt_dir, f"last_{model_name}_{dataset_tag}.ckpt"))

    cfg = {
        "run_name": run_name,
        "model_name": model_name,
        "dataset_tag": dataset_tag,
        "hparams": hparams,
        "patience": patience,
        "max_epochs": max_epochs,
        "resume": resume,
        "init_ckpt_path": init_ckpt_path,
        "last_ckpt": last_out,
        "metrics_csv": os.path.join(logger.log_dir, "metrics.csv"),
        "time_sec": t1 - t0
    }
    cfg_path = os.path.join(run_dir, "run_config.json")
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"[INFO] Wrote config: {cfg_path}")

    def _as_float(x):
        try:
            return float(x.item()) if hasattr(x, "item") else float(x)
        except Exception:
            return None

    final_val_iou = _as_float(trainer.callback_metrics.get("val_iou"))
    best_val_iou  = _as_float(ckpt_cb.best_model_score)
    best_path     = ckpt_cb.best_model_path

    if final_val_iou is not None:
        print(f"[RESULT] Final val_iou: {final_val_iou:.6f}")
    else:
        print("[RESULT] Final val_iou: N/A (no metric logged)")

    if best_val_iou is not None and best_path:
        print(f"[RESULT] Best val_iou : {best_val_iou:.6f}  | best.ckpt: {best_path}")
    else:
        print("[RESULT] Best val_iou : N/A")

    return {
        "run_dir": run_dir,
        "last_ckpt": last_out,
        "metrics_csv": cfg["metrics_csv"],
        "config": cfg_path,
        "final_val_iou": float(trainer.callback_metrics["val_iou"].item()) if "val_iou" in trainer.callback_metrics else None

    }

In [ ]:
hparams = dict(use_pretrained=None, lr=0.00020177966850933734)

<center><font size="5" color="#2D6A4F"><b>Model1600_4</b></font></center>

**group 1**

**Save/Load Processed Data + Lightning DataModule**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
x_train, y_train, x_val, y_val = load_processed_train_valid_vit("1600_4")

save_path = "/content/drive/MyDrive/TranSpeckle/cache/1600_4/processed.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    'x_train': x_train,
    'y_train': y_train,
    'x_valid': x_val,
    'y_valid': y_val
}, save_path)

print(f"Saved to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/1600_4/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_1600_4 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Model training**

In [ ]:
# Optional: Fixed training with best hyperparameters on a different dataset size.
# Skip this step to load the provided trained weights for evaluation/inference.

'''
out = train_fixed_run_dm(
    model_name="TransUNet_B32",
    dataset_tag="1600_4",
    model_class=LitTransUNet,
    hparams=hparams,
    dm=dm_1600_4,
    base_log_dir=base_log_dir,
    max_epochs=500,
    resume="fresh"
)
'''

**Create predictions and evaluation**

In [ ]:
dataset_tag="1600_4"
ckpt_path = f"{base_log_dir}/{model_name}/fixed_runs/{model_name}_{dataset_tag}/checkpoints/last_{model_name}_{dataset_tag}.ckpt"

y_pred = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test1, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test1)

results = evaluate_predictions_torchmetrics(y_pred=y_pred, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])

In [ ]:
# Display a reconstruction example (index 0–1999)

show_prediction_example_pt(x_test1, y_test1, y_pred, index=0)

<center><font size="5" color="#2D6A4F"><b>Model1600_2</b></font></center>

**group 1**

**Save/Load Processed Data + Lightning DataModule**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
x_train, y_train, x_val, y_val = load_processed_train_valid_vit("1600_2")

save_path = "/content/drive/MyDrive/TranSpeckle/cache/1600_2/processed.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    'x_train': x_train,
    'y_train': y_train,
    'x_valid': x_val,
    'y_valid': y_val
}, save_path)

print(f"Saved to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/1600_2/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_1600_2 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Model training**

In [ ]:
# Optional: Fixed training with best hyperparameters on a different dataset size.
# Skip this step to load the provided trained weights for evaluation/inference.

'''
out = train_fixed_run_dm(
    model_name="TransUNet_B32",
    dataset_tag="1600_2",
    model_class=LitTransUNet,
    hparams=hparams,
    dm=dm_1600_2,
    base_log_dir=base_log_dir,
    max_epochs=500,
    resume="fresh"
)
'''

**Create predictions and evaluation**

In [ ]:
dataset_tag="1600_2"
ckpt_path = f"{base_log_dir}/{model_name}/fixed_runs/{model_name}_{dataset_tag}/checkpoints/last_{model_name}_{dataset_tag}.ckpt"

y_pred = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test1, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test1)

results = evaluate_predictions_torchmetrics(y_pred=y_pred, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])

In [ ]:
# Display a reconstruction example (index 0–1999)

show_prediction_example_pt(x_test1, y_test1, y_pred, index=0)

<center><font size="5" color="#2D6A4F"><b>Model800_4</b></font></center>

**group 1**

**Save/Load Processed Data + Lightning DataModule**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
x_train, y_train, x_val, y_val = load_processed_train_valid_vit("800_4")

save_path = "/content/drive/MyDrive/TranSpeckle/cache/800_4/processed.pt"

import os
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    'x_train': x_train,
    'y_train': y_train,
    'x_valid': x_val,
    'y_valid': y_val
}, save_path)

print(f"Saved to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/800_4/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_800_4 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Model training**

In [ ]:
# Optional: Fixed training with best hyperparameters on a different dataset size.
# Skip this step to load the provided trained weights for evaluation/inference.

'''
out = train_fixed_run_dm(
    model_name="TransUNet_B32",
    dataset_tag="800_4",
    model_class=LitTransUNet,
    hparams=hparams,
    dm=dm_800_4,
    base_log_dir=base_log_dir,
    max_epochs=500,
    resume="fresh"
)
'''

In [ ]:
dataset_tag="800_4"
ckpt_path = f"{base_log_dir}/{model_name}/fixed_runs/{model_name}_{dataset_tag}/checkpoints/last_{model_name}_{dataset_tag}.ckpt"

y_pred = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test1, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test1)

results = evaluate_predictions_torchmetrics(y_pred=y_pred, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])

In [ ]:
# Display a reconstruction example (index 0–1999)

show_prediction_example_pt(x_test1, y_test1, y_pred, index=0)

<center><font size="5" color="#2D6A4F"><b>Model800_2</b></font></center>

**group 1**

**Save/Load Processed Data + Lightning DataModule**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
x_train, y_train, x_val, y_val = load_processed_train_valid_vit("800_2")

save_path = "/content/drive/MyDrive/TranSpeckle/cache/800_2/processed.pt"

import os
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    'x_train': x_train,
    'y_train': y_train,
    'x_valid': x_val,
    'y_valid': y_val
}, save_path)

print(f"Saved to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/800_2/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_800_2 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Model training**

In [ ]:
# Optional: Fixed training with best hyperparameters on a different dataset size.
# Skip this step to load the provided trained weights for evaluation/inference.

'''
out = train_fixed_run_dm(
    model_name="TransUNet_B32",
    dataset_tag="800_2",
    model_class=LitTransUNet,
    hparams=hparams,
    dm=dm_800_2,
    base_log_dir=base_log_dir,
    max_epochs=500,
    resume="fresh"
)
'''

In [ ]:
dataset_tag="800_2"
ckpt_path = f"{base_log_dir}/{model_name}/fixed_runs/{model_name}_{dataset_tag}/checkpoints/last_{model_name}_{dataset_tag}.ckpt"

y_pred = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test1, batch_size=100, out_size=(256, 256))


def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:   # [N,2,H,W] -> [N,H,W,2]
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test1)

results = evaluate_predictions_torchmetrics(y_pred=y_pred, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])

In [ ]:
# Display a reconstruction example (index 0–1999)

show_prediction_example_pt(x_test1, y_test1, y_pred, index=0)

<center><font size="5" color="#2D6A4F"><b>Model800_1</b></font></center>

**group 1**

**Save/Load Processed Data + Lightning DataModule**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
x_train, y_train, x_val, y_val = load_processed_train_valid_vit("800_1")

save_path = "/content/drive/MyDrive/TranSpeckle/cache/800_1/processed.pt"

import os
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    'x_train': x_train,
    'y_train': y_train,
    'x_valid': x_val,
    'y_valid': y_val
}, save_path)

print(f"Saved to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/800_1/processed.pt")
x_train, y_train, x_val, y_val = data['x_train'], data['y_train'], data['x_valid'], data['y_valid']

In [ ]:
dm_800_1 = SpeckleDataModule(
    train_data=(x_train, y_train),
    val_data=(x_val, y_val)
)

**Model training**

In [ ]:
# Optional: Fixed training with best hyperparameters on a different dataset size.
# Skip this step to load the provided trained weights for evaluation/inference.

'''
out = train_fixed_run_dm(
    model_name="TransUNet_B32",
    dataset_tag="800_1",
    model_class=LitTransUNet,
    hparams=hparams,
    dm=dm_800_1,
    base_log_dir=base_log_dir,
    max_epochs=500,
    resume="fresh"
)
'''

**Create predictions and evaluation**

In [ ]:
dataset_tag="800_1"
ckpt_path = f"{base_log_dir}/{model_name}/fixed_runs/{model_name}_{dataset_tag}/checkpoints/last_{model_name}_{dataset_tag}.ckpt"

y_pred = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test1, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test1)

results = evaluate_predictions_torchmetrics(y_pred=y_pred, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])

In [ ]:
# Display a reconstruction example (index 0–1999)

show_prediction_example_pt(x_test1, y_test1, y_pred, index=0)

**group4_1**

**Save/Load Processed Data + Lightning DataModule**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
import os, torch
from pathlib import Path

x_test, y_test = load_processed_test_vit("group4_1")


save_path = "/content/drive/MyDrive/TranSpeckle/cache/group4_1/processed.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    "x_test4_1": x_test,
    "y_test4_1": y_test
}, save_path)

print(f"Saved test set to: {save_path}")
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/group4_1/processed.pt")
x_test4_1, y_test4_1 = data['x_test4_1'], data['y_test4_1']

**Create predictions and evaluation**

In [ ]:
dataset_tag="800_1"
ckpt_path = f"{base_log_dir}/{model_name}/fixed_runs/{model_name}_{dataset_tag}/checkpoints/last_{model_name}_{dataset_tag}.ckpt"

y_pred4_1 = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test4_1, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test4_1)

results = evaluate_predictions_torchmetrics(y_pred=y_pred4_1, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])


ckpt_path, trial_no, best_val = get_best_ckpt_path(base_log_dir, model_name)
print(f"[INFO] Using ckpt: {ckpt_path}")
if trial_no is not None:
    print(f"[INFO] From trial #{trial_no} | best val_iou={best_val:.4f}")

In [ ]:
# Display a reconstruction example (index 0–27)

show_prediction_example_pt(x_test4_1, y_test4_1, y_pred4_1, index=0)

**group4_2**

**Save/Load Processed Data + Lightning DataModule**

In [ ]:
# The processed data is already provided in the downloaded folder; this step can be skipped to save runtime, and the cached data can be loaded in the next cell.

'''
import os, torch
from pathlib import Path

x_test, y_test = load_processed_test_vit("group4_2")


save_path = "/content/drive/MyDrive/TranSpeckle/cache/group4_2/processed.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    "x_test4_2": x_test,
    "y_test4_2": y_test
}, save_path)

print(f"Saved test set to: {save_path}")\
'''

In [ ]:
data = torch.load("/content/drive/MyDrive/TranSpeckle/cache/group4_2/processed.pt")
x_test4_2, y_test4_2 = data['x_test4_2'], data['y_test4_2']

**Create predictions and evaluation**

In [ ]:
dataset_tag="800_1"
ckpt_path = f"{base_log_dir}/{model_name}/fixed_runs/{model_name}_{dataset_tag}/checkpoints/last_{model_name}_{dataset_tag}.ckpt"

y_pred4_2 = predict_testset_torch_cls(ckpt_path=ckpt_path, model_class=LitTransUNet, x_test=x_test4_2, batch_size=100, out_size=(256, 256))

def _to_numpy_gt(y):
    y_np = y.numpy() if isinstance(y, torch.Tensor) else y
    if y_np.ndim == 4 and y_np.shape[1] == 2:
        y_np = np.transpose(y_np, (0,2,3,1))
    return y_np

y_true_np = _to_numpy_gt(y_test4_2)

results = evaluate_predictions_torchmetrics(y_pred=y_pred4_2, y_true=y_true_np, threshold=0.5)

print("\n--- Summary ---")
print("Mean IoU:", results["iou_mean"])
print("Mean PCC:", results["pcc_mean"])


ckpt_path, trial_no, best_val = get_best_ckpt_path(base_log_dir, model_name)
print(f"[INFO] Using ckpt: {ckpt_path}")
if trial_no is not None:
    print(f"[INFO] From trial #{trial_no} | best val_iou={best_val:.4f}")

In [ ]:
# Display a reconstruction example (index 0–27)

show_prediction_example_pt(x_test4_2, y_test4_2, y_pred4_2, index=0)